# **Day 78: Command Line Programs with sys.argv and argparse**

A **command line program** is run from the terminal by typing its name and some **arguments** (options).

For example, a GIS tool might be used like this:
```
python ndvi_tool.py --red 0.05 --nir 0.45
```
Many real GIS tools, like **GDAL**, work this way (`gdalinfo image.tif`).

## **sys.argv: The Simple Way**
`sys.argv` is a list of everything typed after `python`. `sys.argv[0]` is the file name.

In [1]:
%%writefile show_args.py
import sys
print("File name :", sys.argv[0])
print("Arguments :", sys.argv[1:])

Writing show_args.py


In [2]:
import subprocess, sys

def run(*args):
    """Run a Python file like in the terminal, and print what it shows."""
    result = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print(result.stdout + result.stderr)

run("show_args.py", "Kolkata", "Siliguri", "--fast")

File name : show_args.py
Arguments : ['Kolkata', 'Siliguri', '--fast']



## **argparse: The Better Way**
`argparse` reads the arguments, **converts** their types, **checks** them, and even makes a **help message** automatically.

## **Example from Remote Sensing: An NDVI Command Line Tool**
Our tool will:
- take the Red and NIR values (required)
- take an optional sensor name, to convert raw numbers (DN) to reflectance
- print the NDVI and the land cover class

In [3]:
%%writefile ndvi_tool.py
"""Calculate NDVI from Red and NIR values."""
import argparse

SCALE = {"reflectance": 1.0, "sentinel2": 0.0001}

def classify(v):
    if v < 0: return "Water"
    if v < 0.2: return "Bare / Built-up"
    if v < 0.5: return "Sparse vegetation"
    return "Dense vegetation"

def main():
    parser = argparse.ArgumentParser(description="NDVI calculator for one pixel")
    parser.add_argument("--red", type=float, required=True, help="Red band value")
    parser.add_argument("--nir", type=float, required=True, help="NIR band value")
    parser.add_argument("--sensor", choices=SCALE.keys(), default="reflectance",
                        help="Type of input values (default: reflectance)")
    parser.add_argument("-v", "--verbose", action="store_true", help="Show more details")
    args = parser.parse_args()

    red = args.red * SCALE[args.sensor]
    nir = args.nir * SCALE[args.sensor]
    if red + nir == 0:
        parser.error("Red + NIR is zero, NDVI cannot be calculated")

    ndvi = (nir - red) / (nir + red)
    if args.verbose:
        print(f"[details] sensor={args.sensor} red={red:.4f} nir={nir:.4f}")
    print(f"NDVI = {ndvi:.3f} -> {classify(ndvi)}")

if __name__ == "__main__":
    main()

Writing ndvi_tool.py


In [4]:
run("ndvi_tool.py", "--help")

usage: ndvi_tool.py [-h] --red RED --nir NIR
                    [--sensor {reflectance,sentinel2}] [-v]

NDVI calculator for one pixel

options:
  -h, --help            show this help message and exit
  --red RED             Red band value
  --nir NIR             NIR band value
  --sensor {reflectance,sentinel2}
                        Type of input values (default: reflectance)
  -v, --verbose         Show more details



In [5]:
run("ndvi_tool.py", "--red", "0.05", "--nir", "0.45")
run("ndvi_tool.py", "--red", "600", "--nir", "3800", "--sensor", "sentinel2", "-v")
run("ndvi_tool.py", "--red", "0.06", "--nir", "0.02")

NDVI = 0.800 -> Dense vegetation



[details] sensor=sentinel2 red=0.0600 nir=0.3800
NDVI = 0.727 -> Dense vegetation



NDVI = -0.500 -> Water



Let us understand the code above:
- `type=float` converts the text from the terminal into a number automatically.
- `required=True` means the program stops with a clear message if the value is missing.
- `choices=...` allows only the listed values.
- `action="store_true"` makes a simple on/off switch (`-v`).
- The `--help` message was created **automatically** from our descriptions.

In [6]:
# argparse also handles mistakes for us
run("ndvi_tool.py", "--red", "0.05")                                   # NIR missing
run("ndvi_tool.py", "--red", "abc", "--nir", "0.4")                    # Not a number
run("ndvi_tool.py", "--red", "0.05", "--nir", "0.4", "--sensor", "x")  # Wrong choice

usage: ndvi_tool.py [-h] --red RED --nir NIR
                    [--sensor {reflectance,sentinel2}] [-v]
ndvi_tool.py: error: the following arguments are required: --nir



usage: ndvi_tool.py [-h] --red RED --nir NIR
                    [--sensor {reflectance,sentinel2}] [-v]
ndvi_tool.py: error: argument --red: invalid float value: 'abc'

usage: ndvi_tool.py [-h] --red RED --nir NIR
                    [--sensor {reflectance,sentinel2}] [-v]
ndvi_tool.py: error: argument --sensor: invalid choice: 'x' (choose from 'reflectance', 'sentinel2')



## **Exit Codes**
When a program ends, it gives an **exit code**:
- **0** means success.
- **Any other number** means an error.

Other programs (and automatic scripts) use this code to know if our program worked.

In [7]:
%%writefile check_file.py
import sys
from pathlib import Path

if len(sys.argv) < 2:
    print("Usage: check_file.py FILE", file=sys.stderr)
    sys.exit(2)
path = Path(sys.argv[1])
if not path.exists():
    print(f"{path} not found", file=sys.stderr)
    sys.exit(1)
print(f"{path} exists, size {path.stat().st_size} bytes")

Writing check_file.py


In [8]:
for args in [["check_file.py"], ["check_file.py", "missing.tif"], ["check_file.py", "ndvi_tool.py"]]:
    result = subprocess.run([sys.executable, *args], capture_output=True, text=True)
    print("Exit code", result.returncode, "|", (result.stdout or result.stderr).strip())

Exit code 2 | Usage: check_file.py FILE


Exit code 1 | missing.tif not found
Exit code 0 | ndvi_tool.py exists, size 1245 bytes


In [9]:
# Quick question
# What exit code means "success"?